# 도서 목록 데이터 추출 및 구조화 1 - 39쪽

In [2]:
!pip install bs4

  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)


In [59]:
from bs4 import BeautifulSoup
import requests

html_books = """
<div class='library'>
    <div class='book' id='b-101'>
        <h3 class='book-title'>딥러닝 텐서플로 마스터</h3>
            <p class='book-author'>김작가</p>
    </div>
    <div class='book' id='b-102'>
        <h3 class='book-title'>이것이 진짜 R 프로그래밍</h3>
            <p class='book-author'>박통계</p>
    </div>
    <div class='book' id='b-103'>
        <h3 class='book-title'>자연어 처리 입문</h3>
            <p class='book-author'>최이과</p>
    </div>
</div>
"""
#담을 것 정의
books_info = []

#파싱
soup = BeautifulSoup(html_books, 'html.parser')

In [60]:
#books로 부터 title, author 추출하기
books = soup.select('.book')

for book in books:
    title = book.select_one('.book-title').text
    author = book.select_one('.book-author').text

    books_info.append({'title': title, 'author': author})

print("=== 추출된 도서 목록 ===")
print(books_info)

=== 추출된 도서 목록 ===
[{'title': '딥러닝 텐서플로 마스터', 'author': '김작가'}, {'title': '이것이 진짜 R 프로그래밍', 'author': '박통계'}, {'title': '자연어 처리 입문', 'author': '최이과'}]


# 도서 목록 데이터 추출 및 구조화 2 - 40쪽

In [58]:
html_news = """
<ul class='news-section'>
    <li class='news-item'><a href='https://news.example.com/ai-1'>인공지능 규제법 통과, IT 업계 파장</a></li>
    <li class='news-item'><a href='https://news.example.com/dev-2'>파이썬 3.12 릴리즈, 무엇이 달라졌나?</a></li>
    <li class='news-item'><a href='https://news.example.com/sec-3'>오픈소스 라이브러리 보안 취약점 경고</a></li>
</ul>
"""

#내용을 담을 리스트 정리
news_list = []

#parsing
soup = BeautifulSoup(html_news, 'html.parser')

#class 안에서 headline, url 추출하기
news = soup.select('.news-item')

for news_1 in news:
    headline = news_1.text
    url = news_1.select_one('a')['href']
    
    #리스트에 key, value형식으로 담기
    news_list.append({'headline': headline, 'url': url})

print("=== 수집된 뉴스 헤드라인 ===")
print(news_list)

=== 수집된 뉴스 헤드라인 ===
[{'headline': '인공지능 규제법 통과, IT 업계 파장', 'url': 'https://news.example.com/ai-1'}, {'headline': '파이썬 3.12 릴리즈, 무엇이 달라졌나?', 'url': 'https://news.example.com/dev-2'}, {'headline': '오픈소스 라이브러리 보안 취약점 경고', 'url': 'https://news.example.com/sec-3'}]


# SQLite 데이터 조작(CRUD) - 65쪽

In [ ]:
import sqlite3

#1. sql 연결해서 테이블 만들기
#1-1. 메모리 DB에 연결(:memory:를 사용하면 생성하지 않고 메모리 상에서만 존재
#connect_sql = sqlite3.connect(':memory:')
connect_sql = sqlite3.connect('client.db')
print("SQL에 연결되었습니다.")

#2. Table을 만들기
data = [
    ('홍길동', 'hong@example.com', 'gold'),
    ('이영희', 'lee@example.com', 'silver'),
    ('김철수', 'kim@example.com', 'gold')
]

#3. cursor로 실행 현황을 메모리에 담기
cursor = connect_sql.cursor()

#4. client.db에 table 만들기 (CREATE)
sql_create = """
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    email TEXT,
    level TEXT
)
"""

#5. client.db에 값 삽입하기 (INSERT)
sql_insert = """
    INSERT INTO users(name, email, level)
    VALUES(?, ?, ?)
"""

#6. 실행하기 및 데이터 담기
cursor.execute(sql_create)
cursor.executemany(sql_insert, data)
connect_sql.commit()

#Gold 사용자 저장할 리스트 정의
gold_users = []

sql_select = """
SELECT * FROM users WHERE level = 'gold'
"""

connect_sql = sqlite3.connect('client.db')

#쿼리 실행하기
cursor.execute(sql_select)

#row (행)의 값 가져와서 출력하기 - fetchall
rows = cursor.fetchall()

for row in rows:
    print(row)
    gold_users.append(row)

print('=== Gold 회원 등급 회원 목록 ===')    
print(gold_users)

connect_sql.close()
print('데이터를 성공적으로 삽입했습니다.')

SQL에 연결되었습니다.
(1, '홍길동', 'hong@example.com', 'gold')
(3, '김철수', 'kim@example.com', 'gold')
=== Gold 회원 등급 회원 목록 ===
[(1, '홍길동', 'hong@example.com', 'gold'), (3, '김철수', 'kim@example.com', 'gold')]
데이터를 성공적으로 삽입했습니다.


# 수집된 데이터프레임의 DB 테이블 저장 및 복원 - 67페이지

In [ ]:
import pandas as pd
import sqlite3

#1. New connection to database
connect_sql_mmb = sqlite3.connect('users.db')
print('SQL에 연결되었습니다.')

#2. 담을 데이터 정의하기
df_members = pd.DataFrame({
    'member_id': ['M01', 'M02', 'M03'],
    'name': ['이민수', '김영희', '박준형'],
    'score': [85, 95, 78]
})

#3. 데이터 삽입하기
df_members.to_sql('members', connect_sql_mmb, index=False)

#4. SQL 끝내기
connect_sql_mmb.close()
print('SQL 연결이 종료되었습니다.')


SQL에 연결되었습니다.
SQL 연결이 종료되었습니다.


In [100]:
#다시 새롭게 조회하기

connect_sql_mmb_2 = sqlite3.connect('users.db')

#Select 쿼리 지정하기
query = 'SELECT * FROM members'
df_read = pd.read_sql(query, connect_sql_mmb_2)

print("=== DB 복원 데이터프레임 ===")
print(df_read)

#검증 후 마지막에 DB 파일 리소스 닫기
connect_sql_mmb_2.close()

=== DB 복원 데이터프레임 ===
  member_id name  score
0       M01  이민수     85
1       M02  김영희     95
2       M03  박준형     78
